# Phase 3 Evaluation Report — Baseline vs Modified GraphRAG

Compares the PDF baseline against our modified GraphRAG on the SAME PubMedQA questions:

| System | Retrieval |
|---|---|
| A. Vector-only | vector search, no graph (ablation anchor) |
| B. Baseline | required expansion + legacy fusion, no PageRank/adaptive |
| C. Fused | baseline + 3-term graph-aware fusion |
| D. PageRank | fused + subgraph-scoped PageRank |
| E. Adaptive (ours) | fused + PageRank + adaptive retrieval |

Metrics (preserved exactly): article-level Recall@5 / Recall@10 / MRR for retrieval; ROUGE-L F1 / BERTScore F1 (vs PubMedQA long answers) for generation.

**Every number below comes from `data/processed/eval_systems.csv`, `eval_comparison.csv`, and `eval_generation.csv` written by `python -m src.evaluation.compare`. Nothing here is fabricated — if the files are absent, the notebook says so instead of showing numbers.**

## Setup

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.evaluation.systems import describe_systems

PROCESSED = Path("data/processed")
for s in describe_systems():
    print(f"{s['label']}: {s['description']}")

## 1. Dataset & questions

Run this after `python -m src.evaluation.compare` (requires populated Neo4j + chunk files; generation additionally needs LLM credentials).

In [ ]:
q_path = PROCESSED / "eval_questions.jsonl"
if q_path.exists():
    n_q = sum(1 for _ in open(q_path, encoding="utf-8"))
    print(f"Evaluation questions: {n_q} (data/processed/eval_questions.jsonl)")
else:
    print("eval_questions.jsonl NOT found — generate it first (see README: evaluation).")

for name in ("semantic_cluster.jsonl", "fixed_token.jsonl"):
    p = PROCESSED / name
    print(f"{name}: {'present' if p.exists() else 'MISSING'}")

## 2. Retrieval results (all systems, same questions)

In [ ]:
systems_csv = PROCESSED / "eval_systems.csv"
if systems_csv.exists():
    systems_df = pd.read_csv(systems_csv)
    display(systems_df)
else:
    print("eval_systems.csv NOT found — run: python -m src.evaluation.compare")

In [ ]:
comp_csv = PROCESSED / "eval_comparison.csv"
if comp_csv.exists():
    display(pd.read_csv(comp_csv))
else:
    print("eval_comparison.csv NOT found — run: python -m src.evaluation.compare")

## 3. Retrieval chart

In [ ]:
from IPython.display import Image
chart = PROCESSED / "retrieval_comparison.png"
if chart.exists():
    display(Image(filename=str(chart)))
else:
    print("retrieval_comparison.png NOT found — run: python -m src.evaluation.compare")

## 4. Generation results (baseline vs modified)

In [ ]:
gen_csv = PROCESSED / "eval_generation.csv"
if gen_csv.exists():
    display(pd.read_csv(gen_csv))
else:
    print("eval_generation.csv NOT found — generation needs LLM credentials; see README.")

## 5. Ablation reading guide

Compare adjacent columns of `eval_systems.csv`: vector_only → baseline isolates graph expansion; baseline → fused isolates the 3-term fusion; fused → pagerank isolates PageRank; pagerank → adaptive isolates the adaptive decision. A delta favoring the later system on the SAME questions suggests that component changes behavior; with ~200 questions, treat small deltas as inconclusive.

## 6. Interpretation (fill in ONLY from measured tables above)

1. Does graph expansion change Recall@5 / Recall@10 vs vector-only on these questions?
2. What is the MRR delta, and what does it imply for ranking quality?
3. Do ROUGE-L / BERTScore differ between baseline and modified generation contexts?
4. Which adaptive decisions (vector-focused vs graph) dominate, and on what queries?

## 7. Limitations

- Article-level binary recall rewards finding the gold article, not the best chunk.
- Generation quality depends on the LLM backend/credentials as well as retrieval.
- PageRank results require the GDS plugin; without it the pipeline degrades gracefully and D ≈ C.
- No superiority claim is made here beyond the measured deltas.